# Notebook 4: Agente con Herramientas (ReAct Loop)

## ¿De qué trata este notebook?

Este es el notebook más avanzado de la serie. Combina todo lo aprendido en los anteriores y añade el concepto más potente: **herramientas**.

Hasta ahora el LLM solo generaba texto. En este notebook le damos **herramientas** (tools): funciones Python que el modelo puede decidir llamar cuando necesite información o realizar acciones.

### La analogía del detective

Un detective no adivina: razona, pide informes, hace preguntas, y vuelve a razonar con la nueva información.

```
Razona: "Necesito saber el clima de Madrid"
    ↓
Actúa: llama a la herramienta get_clima("Madrid")
    ↓
Observa: "18°C, parcialmente nublado"
    ↓
Razona: "Ahora sé el clima, puedo responder"
    ↓
Responde: "El clima en Madrid es 18°C..."
```

Este patrón (Razonar → Actuar → Observar → Razonar) se llama **ReAct** (de Reason + Act) y es el estándar de la industria para agentes de IA.

### El ciclo del grafo

```
START → agente → ¿hay tool call? → sí → ejecutar_tools → agente (de nuevo)
                               → no → END
```

El grafo puede dar varias vueltas antes de terminar. Esto es nuevo respecto a los notebooks anteriores donde el flujo siempre era lineal.

### Conceptos nuevos en este notebook

| Concepto | ¿Qué es? |
|----------|---------|
| **@tool** | Decorador que convierte una función Python en herramienta del LLM |
| **bind_tools()** | Conecta las tools al LLM para que las pueda invocar |
| **tool_calls** | Campo en la respuesta del LLM cuando decide usar una herramienta |
| **ToolNode** | Nodo prebuilt que ejecuta automáticamente los tool_calls |
| **Ciclo ReAct** | El bucle `agente → tools → agente → ...` hasta respuesta final |
| **create_react_agent** | Helper que construye el grafo ReAct completo en una línea |

---
## 1. Instalación y configuración

In [1]:
# %pip install -qU langgraph langchain-google-genai langchain-core

In [2]:
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="C:/Users/Oscar/OneDrive - FM4/Escritorio/EVOLVE/Data Science/EVOLVE/Victor_Prior_IA_Generativa/Proyecto_Modulo_IAGen/.env")
API_KEY = os.getenv("GEMINI_API_KEY_001")

# API key de Gemini
# API_KEY = userdata.get('GEMINI_API_KEY')

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0.7,
    api_key = API_KEY
)
print("Modelo listo:", llm.model)

Modelo listo: gemini-2.5-flash-lite


---
## 2. El patrón ReAct explicado

ReAct = **Re**ason + **Act**

```
Pensamiento: "Necesito saber el clima de Madrid"
    ↓
Acción: llamar tool get_clima(ciudad="Madrid")
    ↓
Observación: "18°C, parcialmente nublado"
    ↓
Pensamiento: "Ahora sé el clima, puedo responder"
    ↓
Respuesta Final: "El clima en Madrid es 18°C..."
```

En LangGraph esto se implementa como un bucle:

```
START → agente → ¿tool call? → sí → ejecutar_tools → agente → ...
                             → no → END
```

---
## 3. Definir las herramientas (Tools)

### ¿Qué es un tool y cómo lo entiende el LLM?

Un tool es simplemente una función Python con un decorador especial (`@tool`). El LLM no ejecuta la función directamente — lo que hace es:
1. Leer el **docstring** (el texto entre `"""`) para entender qué hace la función
2. Decidir si la necesita para responder
3. Si la necesita, devolver un `tool_call` con el nombre de la función y los argumentos

El código Python (no el LLM) ejecuta la función real y devuelve el resultado al LLM.

**El docstring es crítico:** si está mal escrito, el LLM no sabrá cuándo usarlo o con qué argumentos.

### Las 3 herramientas de este notebook

| Tool | ¿Qué hace? | Datos reales |
|------|-----------|-------------|
| `calcular` | Evalúa expresiones matemáticas | Sí (usa la librería `math` de Python) |
| `obtener_clima` | Consulta el clima de una ciudad | No (datos ficticios hardcoded) |
| `buscar_wikipedia` | Busca información sobre un tema | No (datos ficticios hardcoded) |

In [4]:
from langchain_core.tools import tool
import math
import random

@tool
def calcular(expresion: str) -> str:
    """Evalúa una expresión matemática. Usa esto para cualquier cálculo numérico.
    Soporta operaciones básicas (+, -, *, /), potencias (**), y funciones
    matemáticas como sqrt, sin, cos, log.
    Ejemplo: '2 ** 10', 'math.sqrt(144)', '15 * 7 + 3'
    """
    try:
        # Evaluamos de forma segura con math disponible
        resultado = eval(expresion, {"__builtins__": {}}, {"math": math, "sqrt": math.sqrt})
        return f"Resultado: {resultado}"
    except Exception as e:
        return f"Error al calcular '{expresion}': {e}"


@tool
def obtener_clima(ciudad: str) -> str:
    """Obtiene el clima actual de una ciudad.
    Retorna temperatura, condición y humedad.
    Úsalo cuando el usuario pregunte por el tiempo o clima en algún lugar.
    """
    # Simulamos una API de clima con datos ficticios
    climas_falsos = {
        "madrid": {"temp": 22, "condicion": "Soleado",          "humedad": 40},
        "barcelona": {"temp": 19, "condicion": "Parcialmente nublado", "humedad": 65},
        "sevilla": {"temp": 29, "condicion": "Muy caluroso",    "humedad": 30},
        "bilbao": {"temp": 15, "condicion": "Lluvioso",         "humedad": 85},
        "default": {"temp": random.randint(10, 30), "condicion": "Variable", "humedad": 60}
    }
    datos = climas_falsos.get(ciudad.lower(), climas_falsos["default"])
    return (f"Clima en {ciudad}: {datos['temp']}°C, {datos['condicion']}, "
            f"humedad {datos['humedad']}%")


@tool
def buscar_wikipedia(tema: str) -> str:
    """Busca información sobre un tema en Wikipedia.
    Retorna un resumen conciso. Úsalo cuando necesites información factual
    sobre personas, lugares, eventos históricos o conceptos.
    """
    # Simulamos Wikipedia con datos hardcoded
    base_datos = {
        "python": "Python es un lenguaje de programación de alto nivel creado por Guido van Rossum en 1991. Es conocido por su sintaxis clara y su enfoque en la legibilidad.",
        "madrid": "Madrid es la capital y ciudad más grande de España, con una población de más de 3 millones de habitantes. Es conocida por el Museo del Prado y el estadio Santiago Bernabéu.",
        "einstein": "Albert Einstein (1879–1955) fue un físico alemán-estadounidense que desarrolló la teoría de la relatividad. Ganó el Premio Nobel de Física en 1921.",
        "langgraph": "LangGraph es un framework de Python para construir agentes y flujos de trabajo con LLMs usando grafos de estado. Fue creado por LangChain Inc."
    }
    resultado = base_datos.get(tema.lower())
    if resultado:
        return resultado
    return f"Información sobre '{tema}': Tema encontrado con información general. Es un concepto/entidad relevante en su dominio."


# Lista de tools disponibles
tools = [calcular, obtener_clima, buscar_wikipedia]

print(f"✅ {len(tools)} tools definidas:")
for t in tools:
    print(f"  - {t.name}: {t.description[:60]}...")

✅ 3 tools definidas:
  - calcular: Evalúa una expresión matemática. Usa esto para cualquier cál...
  - obtener_clima: Obtiene el clima actual de una ciudad.
    Retorna temperatu...
  - buscar_wikipedia: Busca información sobre un tema en Wikipedia.
    Retorna un...


---
## 4. Conectar el LLM con los tools

### ¿Qué hace `bind_tools`?

`bind_tools` le dice al LLM qué herramientas tiene disponibles. A partir de este momento, cuando el LLM reciba una pregunta, puede responder de dos maneras:
1. **Con texto:** respuesta directa al usuario (ha terminado)
2. **Con `tool_calls`:** lista de herramientas que quiere ejecutar (no ha terminado aún)

Esta celda hace una prueba directa para verificar que el LLM entiende cuándo debe usar la herramienta `calcular`.

In [5]:
# LLM con tools "atadas"
llm_con_tools = llm.bind_tools(tools)

# Probemos cómo reacciona el LLM cuando le hacemos una pregunta que requiere una tool
from langchain_core.messages import HumanMessage

mensaje_prueba = HumanMessage(content="¿Cuánto es 2 elevado a la 10?")
respuesta_prueba = llm_con_tools.invoke([mensaje_prueba])

print("¿El LLM quiere usar una tool?", len(respuesta_prueba.tool_calls) > 0)
if respuesta_prueba.tool_calls:
    print("Tool call solicitada:")
    for tc in respuesta_prueba.tool_calls:
        print(f"  Nombre: {tc['name']}")
        print(f"  Args:   {tc['args']}")

¿El LLM quiere usar una tool? True
Tool call solicitada:
  Nombre: calcular
  Args:   {'expresion': '2 ** 10'}


---
## 5. Definir el Estado y los Nodos

### El estado

El estado de este agente es lo más simple posible: solo una lista de mensajes con el reducer `add_messages` que ya conocemos del Notebook 3.

Cada vuelta del ciclo ReAct añade mensajes a esa lista:
- El mensaje del usuario
- El `AIMessage` del agente (con o sin `tool_calls`)
- El `ToolMessage` con el resultado de ejecutar la herramienta
- Otro `AIMessage` del agente (ya con la respuesta final)

In [6]:
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage

class EstadoAgente(TypedDict):
    mensajes: Annotated[list[BaseMessage], add_messages]

### El nodo agente

Este nodo es el cerebro. En cada vuelta del ciclo:
1. Recibe todos los mensajes acumulados (incluyendo los resultados de tools anteriores)
2. Llama al LLM con `llm_con_tools`
3. Si el LLM devuelve `tool_calls` → el router mandará el flujo a `ToolNode`
4. Si el LLM devuelve texto → el router terminará el grafo

In [7]:
from langchain_core.messages import SystemMessage

SYSTEM_REACT = """Eres un asistente inteligente con acceso a herramientas.
Razona paso a paso y usa las herramientas disponibles cuando sea necesario.
Para preguntas que requieran cálculos, usa la tool 'calcular'.
Para clima, usa 'obtener_clima'. Para información factual, usa 'buscar_wikipedia'."""

def nodo_agente(estado: EstadoAgente) -> dict:
    """
    El nodo principal del agente. Llama al LLM y puede:
    1. Retornar tool_calls → el grafo ejecutará las tools
    2. Retornar texto final → el grafo termina
    """
    mensajes = [SystemMessage(content=SYSTEM_REACT)] + estado["mensajes"]
    respuesta = llm_con_tools.invoke(mensajes)

    if respuesta.tool_calls:
        print(f"[Agente] Solicitando {len(respuesta.tool_calls)} tool(s):")
        for tc in respuesta.tool_calls:
            print(f"  → {tc['name']}({tc['args']})")
    else:
        print("[Agente] Generando respuesta final (sin más tools)")

    return {"mensajes": [respuesta]}

print("Nodo agente definido")

Nodo agente definido


---
## 6. ToolNode: el ejecutor de herramientas

### ¿Qué es ToolNode?

`ToolNode` es un nodo ya construido que viene incluido en LangGraph. No tienes que escribirlo tú.

Lo que hace automáticamente:
1. Lee el último `AIMessage` del estado
2. Mira qué `tool_calls` tiene (qué herramientas quiere llamar el LLM)
3. Ejecuta cada herramienta con los argumentos indicados
4. Devuelve los resultados como `ToolMessage` (que el reducer añadirá al historial)

Después de `ToolNode`, el flujo vuelve al nodo agente para que el LLM vea los resultados y decida si responde o necesita más herramientas.

In [8]:
from langgraph.prebuilt import ToolNode

# ToolNode ejecuta automáticamente los tool_calls del agente
nodo_tools = ToolNode(tools)

print("ToolNode configurado con:", [t.name for t in tools])

ToolNode configurado con: ['calcular', 'obtener_clima', 'buscar_wikipedia']


---
## 7. La función de routing del ReAct loop

### ¿Qué decide el router aquí?

Esta función es la que crea el bucle. Después de que el nodo agente responde, el router mira el último mensaje:

- **Si tiene `tool_calls`** → hay herramientas pendientes de ejecutar → ir a `ToolNode`
- **Si no tiene `tool_calls`** → el LLM ya generó la respuesta final → terminar

Esta decisión puede repetirse muchas veces. Una pregunta compleja puede requerir 3 o 4 vueltas del bucle antes de que el LLM tenga toda la información que necesita.

In [9]:
from typing import Literal

def router_react(estado: EstadoAgente) -> Literal["tools", "__end__"]:
    """
    Si el último mensaje del agente tiene tool_calls → ejecutar tools
    Si no → terminar (respuesta final generada)
    """
    ultimo_mensaje = estado["mensajes"][-1]

    if hasattr(ultimo_mensaje, "tool_calls") and ultimo_mensaje.tool_calls:
        return "tools"    # Continúa el ciclo
    return "__end__"      # Termina el grafo

---
## 8. Construir el grafo ReAct

### El grafo con bucle

Este grafo tiene algo que no tenían los anteriores: un **ciclo**. La flecha de `tools → agente` crea un bucle que puede repetirse indefinidamente.

```
START → agente → (router)
                   ├── "tools" → ToolNode → agente (de nuevo)
                   └── "__end__" → END
```

`MemorySaver` lo añadimos aquí también para poder inspeccionar el historial completo de mensajes después de la ejecución.

In [10]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

grafo = StateGraph(EstadoAgente)

# Nodos
grafo.add_node("agente", nodo_agente)
grafo.add_node("tools",  nodo_tools)

# El agente siempre empieza primero
grafo.add_edge(START, "agente")

# Después del agente: ¿hay tool calls? → ir a tools o terminar
grafo.add_conditional_edges(
    "agente",
    router_react,
    {"tools": "tools", "__end__": END}
)

# Después de ejecutar tools: siempre volver al agente (el ciclo)
grafo.add_edge("tools", "agente")

app = grafo.compile(checkpointer=MemorySaver())
print("✅ Agente ReAct compilado")

✅ Agente ReAct compilado


In [11]:
# Visualizar el ciclo
print(app.get_graph().draw_ascii())

        +-----------+         
        | __start__ |         
        +-----------+         
               *              
               *              
               *              
          +--------+          
          | agente |          
          +--------+          
          .         .         
        ..           ..       
       .               .      
+---------+         +-------+ 
| __end__ |         | tools | 
+---------+         +-------+ 


---
## 9. Probar el agente

La función `preguntar_agente` envía una pregunta al agente y muestra la respuesta final. Notar que `thread_id` único por prueba garantiza que no se mezclen los historiales.

**Lo que entra:** texto de la pregunta  
**Lo que sale:** la respuesta final del agente (después de todas las vueltas del bucle ReAct)

In [12]:
def preguntar_agente(pregunta: str, thread_id: str = "test-001"):
    config = {"configurable": {"thread_id": thread_id}}
    print("\n" + "=" * 65)
    print(f"PREGUNTA: {pregunta}")
    print("=" * 65)

    resultado = app.invoke(
        {"mensajes": [HumanMessage(content=pregunta)]},
        config=config
    )

    respuesta_final = resultado["mensajes"][-1].content
    print(f"\nRESPUESTA FINAL:\n{respuesta_final}")
    return resultado

In [15]:
# Prueba 1: Cálculo (usa la tool 'calcular')
r1 = preguntar_agente("¿Cuánto es la raíz cuadrada de 1764?", "thread-calculo")


PREGUNTA: ¿Cuánto es la raíz cuadrada de 1764?
[Agente] Generando respuesta final (sin más tools)

RESPUESTA FINAL:
42.0


In [20]:
# Prueba 2: Clima (usa la tool 'obtener_clima')
r2 = preguntar_agente("¿Qué temperatura hace hoy en Sevilla?", "thread-clima")


PREGUNTA: ¿Qué temperatura hace hoy en Sevilla?
[Agente] Solicitando 1 tool(s):
  → obtener_clima({'ciudad': 'Sevilla'})


ValueError: No message found in input

In [21]:
# Prueba 3: Multi-step (usa varias tools en secuencia)
r3 = preguntar_agente(
    "Busca info sobre Madrid y dime también su clima de hoy. "
    "Además, ¿cuánto es 3 elevado a la 8?",
    "thread-multistep"
)


PREGUNTA: Busca info sobre Madrid y dime también su clima de hoy. Además, ¿cuánto es 3 elevado a la 8?
[Agente] Solicitando 3 tool(s):
  → buscar_wikipedia({'tema': 'Madrid'})
  → obtener_clima({'ciudad': 'Madrid'})
  → calcular({'expresion': '3 ** 8'})


ValueError: No message found in input

---
## 10. Ver el trace completo del razonamiento

### ¿Qué es el trace?

El trace es la secuencia completa de mensajes que se acumularon durante el ciclo ReAct. Muestra exactamente qué hizo el agente en cada vuelta:

1. `HumanMessage`: la pregunta inicial
2. `AIMessage` con `tool_calls`: el agente decide qué herramienta usar
3. `ToolMessage`: el resultado de ejecutar la herramienta
4. `AIMessage` con `tool_calls`: (si necesita más herramientas, otro paso)
5. `AIMessage` con texto: la respuesta final

Inspeccionar el trace es la forma más efectiva de depurar el comportamiento de un agente.

In [22]:
# Ver todos los mensajes del ciclo ReAct para el thread multistep
from langchain_core.messages import AIMessage, ToolMessage

config_multi = {"configurable": {"thread_id": "thread-multistep"}}
estado_final = app.get_state(config_multi)

print("TRACE COMPLETO DEL RAZONAMIENTO:")
print("=" * 65)

for i, msg in enumerate(estado_final.values["mensajes"]):
    tipo = type(msg).__name__
    print(f"\n[{i+1}] {tipo}:")

    if isinstance(msg, HumanMessage):
        print(f"  → {msg.content}")
    elif isinstance(msg, AIMessage):
        if msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  → TOOL CALL: {tc['name']}({tc['args']})")
        else:
            print(f"  → RESPUESTA: {msg.content[:100]}..." if len(msg.content) > 100 else f"  → RESPUESTA: {msg.content}")
    elif isinstance(msg, ToolMessage):
        print(f"  → RESULTADO: {msg.content}")

TRACE COMPLETO DEL RAZONAMIENTO:

[1] HumanMessage:
  → Busca info sobre Madrid y dime también su clima de hoy. Además, ¿cuánto es 3 elevado a la 8?

[2] AIMessage:
  → TOOL CALL: buscar_wikipedia({'tema': 'Madrid'})
  → TOOL CALL: obtener_clima({'ciudad': 'Madrid'})
  → TOOL CALL: calcular({'expresion': '3 ** 8'})


---
## 11. Versión simplificada con create_react_agent

### ¿Para qué sirve `create_react_agent`?

Todo lo que construimos manualmente (el estado, el nodo agente, el ToolNode, el router, la compilación) es un patrón tan común que LangGraph lo tiene empaquetado en una sola función.

`create_react_agent` construye el grafo ReAct completo automáticamente. El resultado es funcionalmente equivalente al grafo que construimos a mano.

**¿Cuándo usar cada versión?**
- **Manual:** cuando necesitas personalizar alguna parte (el estado, el routing, añadir nodos extra)
- **`create_react_agent`:** cuando el patrón estándar es suficiente y quieres código más corto

In [23]:
from langgraph.prebuilt import create_react_agent

# Una sola línea en lugar de todo el grafo manual
agente_rapido = create_react_agent(
    model=llm,
    tools=tools,
    prompt=SYSTEM_REACT
)

resultado = agente_rapido.invoke({
    "messages": [HumanMessage(content="¿Cuánto es 42 * 42?")]
})

print("Respuesta con create_react_agent:")
print(resultado["messages"][-1].content)

C:\Users\Oscar\AppData\Local\Temp\ipykernel_10736\3294284086.py:4: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agente_rapido = create_react_agent(


ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash-lite' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 42.542000453s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerM

---
## 12. Ejercicios propuestos

1. **Nueva tool**: Crea una tool `convertir_moneda(cantidad, de_moneda, a_moneda)` que simule conversiones de divisa. Añádela al agente.

2. **Límite de iteraciones**: El grafo podría quedar en un bucle infinito si el LLM siempre quiere usar tools. Añade un campo `iteraciones` al estado y limita el ciclo a máximo 5 vueltas.

3. **Tool con efectos secundarios**: Crea una tool `guardar_nota(titulo, contenido)` que guarde notas en un archivo local. ¿Cómo gestionar los efectos secundarios en un grafo?

## Resumen

| Concepto | Descripción |
|---|---|
| `@tool` | Decora una función Python para convertirla en herramienta del LLM |
| `bind_tools()` | Conecta tools al LLM para que las pueda invocar |
| `tool_calls` | Campo en el AIMessage cuando el LLM quiere usar una tool |
| `ToolNode` | Nodo prebuilt que ejecuta automáticamente los tool_calls |
| Ciclo ReAct | `agente → tools → agente → ...` hasta que no haya más tool_calls |
| `create_react_agent` | Helper que construye el grafo ReAct completo automáticamente |